In [1]:
import pandas as pd
import numpy as np
import pandasql as ps

In [4]:
# Import conversion tables
slot_block = pd.read_csv('../data/slot_block.csv')
validator_index = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_index', 'validator_pubkey'])
validator_metadata = pd.read_csv('../int/validator_metadata_v2.csv', usecols=['validator_index', 'pool', 'category', 'pool_size_label'])

largest_pools = ('Lido','Coinbase','Binance','Rocketpool','Kraken','OKX','Bitcoin Suisse','Ledger Live','Ether.Fi','Mantle')
validator_metadata.loc[~validator_metadata['pool'].isin(largest_pools), 'pool'] = 'Other Stakers'


In [4]:
# Fetch and transform deposits
deposits = pd.read_csv('../data/deposits.csv', usecols=['block_number', 'validator_pubkey', 'amount'])
deposits = deposits.sort_values(by='block_number').reset_index(drop=True)
deposits = pd.merge(deposits, slot_block, left_on='block_number', right_on='block', how='left')
deposits = deposits.drop(columns=['block', 'block_number'])
deposits['validator_pubkey'] = '0' + deposits['validator_pubkey'].str.replace('\\', '')
deposits['amount'] = deposits['amount'] / (10**9)
deposits = pd.merge(deposits, validator_index, on='validator_pubkey', how='left')
deposits.drop('validator_pubkey', axis=1, inplace=True)
deposits = deposits[['slot', 'validator_index', 'amount']]
deposits['slot'] = deposits['slot'].fillna(0)

# Merge validator metadata
deposits = pd.merge(deposits, validator_metadata, on='validator_index', how='left')
deposits = deposits.drop(columns='validator_index')

# Group slots before the cut off date
grouped_deposits = deposits[deposits['slot'] == 0]
grouped_deposits = grouped_deposits.groupby(['slot', 'pool', 'category', 'pool_size_label']).agg({
    'amount': 'sum'
}).reset_index()
remaining_df = deposits[deposits['slot'] != 0]
result_df = pd.concat([grouped_deposits, remaining_df])
result_df = result_df[deposits.columns]

# Calculate cumulative values
deposits = result_df
deposits['pool_cumulative'] = deposits.groupby('pool')['amount'].cumsum()
deposits['category_cumulative'] = deposits.groupby('category')['amount'].cumsum()
deposits['size_cumulative'] = deposits.groupby('pool_size_label')['amount'].cumsum()
deposits['cumulative'] = deposits['amount'].cumsum()
deposits

,slot,amount,pool,category,pool_size_label,pool_cumulative,category_cumulative,size_cumulative,cumulative
0,0.0,993088.0,Binance,CEX,100+,993088.0,993088.0,993088.0,993088.0
1,0.0,290080.0,Bitcoin Suisse,CEX,100+,290080.0,1283168.0,1283168.0,1283168.0
2,0.0,2101344.0,Coinbase,CEX,100+,2101344.0,3384512.0,3384512.0,3384512.0
3,0.0,1145088.0,Kraken,CEX,100+,1145088.0,4529600.0,4529600.0,4529600.0
4,0.0,5632.0,Ledger Live,Staking Pools,100+,5632.0,5632.0,4535232.0,4535232.0
...,...,...,...,...,...,...,...,...,...
1461923,8985382.0,32.0,Other Stakers,CEX,100+,17875006.0,13427681.0,40842530.0,44263082.0
1461924,8985384.0,32.0,Other Stakers,CEX,100+,17875038.0,13427713.0,40842562.0,44263114.0
1461925,8985385.0,32.0,Other Stakers,CEX,100+,17875070.0,13427745.0,40842594.0,44263146.0
1461926,8985387.0,32.0,Other Stakers,CEX,100+,17875102.0,13427777.0,40842626.0,44263178.0


In [5]:
# Fetch, combine, and transform the withdrawals tables
withdrawals = pd.read_csv('../data/withdrawals_slot_under_7m.csv')
withdrawals = pd.concat([withdrawals, pd.read_csv('../data/withdrawals_slot_over_7m.csv')])
withdrawals = withdrawals.sort_values(by='block_number').reset_index(drop=True)
withdrawals.rename(columns={'block_number': 'slot'}, inplace=True)
withdrawals['amount'] = withdrawals['amount'] / (10**9)
withdrawals['slot'] = withdrawals['slot'].fillna(0)
withdrawals.loc[withdrawals['slot'] < 0, 'slot'] = 0

# Merge validator metadata
withdrawals = pd.merge(withdrawals, validator_metadata, on='validator_index', how='left')
withdrawals = withdrawals.drop(columns='validator_index')
withdrawals = withdrawals.groupby(['slot', 'pool', 'category', 'pool_size_label']).agg({
    'amount': 'sum'
}).reset_index()

# Calculate cumulative values
withdrawals['pool_cumulative'] = withdrawals.groupby('pool')['amount'].cumsum()
withdrawals['category_cumulative'] = withdrawals.groupby('category')['amount'].cumsum()
withdrawals['size_cumulative'] = withdrawals.groupby('pool_size_label')['amount'].cumsum()
withdrawals['cumulative'] = withdrawals['amount'].cumsum()
withdrawals


,slot,pool,category,pool_size_label,amount,pool_cumulative,category_cumulative,size_cumulative,cumulative
0,6209540,Other Stakers,Unidentified,1,4.547643,4.547643e+00,4.547643e+00,4.547643e+00,4.547643e+00
1,6209540,Other Stakers,Unidentified,20-99,4.440881,8.988524e+00,8.988524e+00,4.440881e+00,8.988524e+00
2,6209540,Other Stakers,Unidentified,6-19,4.451501,1.344002e+01,1.344002e+01,4.451501e+00,1.344002e+01
3,6209542,Other Stakers,Solo Stakers,100+,4.480416,1.792044e+01,4.480416e+00,4.480416e+00,1.792044e+01
4,6209542,Other Stakers,Solo Stakers,6-19,33.073281,5.099372e+01,3.755370e+01,3.752478e+01,5.099372e+01
...,...,...,...,...,...,...,...,...,...
4872529,8984511,Other Stakers,Unidentified,100+,0.154056,5.687079e+06,3.327629e+06,1.275439e+07,1.368459e+07
4872530,8984511,Other Stakers,Unidentified,2-5,0.018167,5.687079e+06,3.327629e+06,1.455909e+05,1.368459e+07
4872531,8984512,Coinbase,CEX,100+,0.127894,2.508913e+06,5.873813e+06,1.275439e+07,1.368459e+07
4872532,8984512,Other Stakers,Unidentified,100+,0.109813,5.687079e+06,3.327629e+06,1.275439e+07,1.368459e+07


In [1]:
6209540 // 7200 * 7200

6206400

In [6]:
# Fetch and transform rewards
rewards = pd.read_csv('../data/validator_rewards.csv', usecols=['consensus_slot', 'validator_index', 'consensus_rewards', 'execution_rewards'])
rewards = rewards.sort_values(by='consensus_slot')
rewards.rename(columns={'consensus_slot': 'slot'}, inplace=True)
rewards['consensus_rewards'] = rewards['consensus_rewards'] / (10**8)
rewards['execution_rewards'] = rewards['execution_rewards'] / (10**9)
rewards['slot'] = rewards['slot'].fillna(0)
rewards.loc[rewards['slot'] < 0, 'slot'] = 0

# Group slots before the cut off date
rewards = pd.merge(rewards, validator_metadata, on='validator_index', how='left')
rewards = rewards.drop(columns='validator_index')
grouped_rewards = rewards[rewards['slot'] == 0]
grouped_rewards = grouped_rewards.groupby(['slot', 'pool', 'category', 'pool_size_label']).agg({
    'consensus_rewards': 'sum',
    'execution_rewards': 'sum'
}).reset_index()
remaining_df = rewards[rewards['slot'] != 0]
result_df = pd.concat([grouped_rewards, remaining_df])
result_df = result_df[rewards.columns]

# Calculate cumulative values
rewards = result_df
rewards['pool_cumulative'] = rewards.groupby('pool')['consensus_rewards'].cumsum()
rewards['category_cumulative'] = rewards.groupby('category')['consensus_rewards'].cumsum()
rewards['size_cumulative'] = rewards.groupby('pool_size_label')['consensus_rewards'].cumsum()
rewards['cumulative'] = rewards['consensus_rewards'].cumsum()
rewards

,slot,consensus_rewards,execution_rewards,pool,category,pool_size_label,pool_cumulative,category_cumulative,size_cumulative,cumulative
0,0,0.000000,0.000000,Other Stakers,Unidentified,100+,0.000000,0.000000,0.000000e+00,0.000000e+00
1,1,0.000000,0.000000,Other Stakers,Unidentified,1,0.000000,0.000000,0.000000e+00,0.000000e+00
2,2,0.000000,0.000000,Bitcoin Suisse,CEX,100+,0.000000,0.000000,0.000000e+00,0.000000e+00
3,3,0.000000,0.000000,Other Stakers,Unidentified,100+,0.000000,0.000000,0.000000e+00,0.000000e+00
4,4,0.000000,0.000000,Other Stakers,Unidentified,100+,0.000000,0.000000,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
8986171,8986171,0.443984,0.018265,Ledger Live,Staking Pools,100+,16497.339413,118263.388813,1.895075e+06,2.096238e+06
8986172,8986172,0.441031,0.020240,Kraken,CEX,100+,122871.533251,714506.312262,1.895076e+06,2.096238e+06
8986173,8986173,0.445289,0.022277,Lido,Liquid Staking,100+,624980.222626,710885.724773,1.895076e+06,2.096239e+06
8986174,8986174,0.421257,0.009399,Other Stakers,Unidentified,20-99,751875.980159,462504.820037,9.946786e+04,2.096239e+06


In [7]:
slots_df = pd.DataFrame({'slot': range(1200, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200
deposits_df = deposits[deposits['slot'].isin(slots_df['slot'])]
withdrawals_df = withdrawals[withdrawals['slot'].isin(slots_df['slot'])]
rewards_df = rewards[rewards['slot'].isin(slots_df['slot'])]

In [8]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(1200, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200

slots_df['pool_size_label'] = np.nan
slots_df['size_cumulative'] = np.nan

# Define the columns for the correct unique values in your dataset
columns = rewards['pool_size_label'].unique()

# Assign values from columns to pool_size_label column cyclically
slots_df['pool_size_label'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Create the pivot table with pool_size_label as columns and size_cumulative as values
slots_pivot = slots_df.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative', dropna=False)

deposits_raw = deposits_df.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative')
withdrawals_raw = withdrawals_df.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative')
rewards_raw = rewards_df.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative')

# Combine deposits_pivot to slots_pivot by slot index
deposits_pivot = slots_pivot.combine_first(deposits_raw).ffill()
withdrawals_pivot = slots_pivot.combine_first(withdrawals_raw).ffill()
rewards_pivot = slots_pivot.combine_first(rewards_raw).ffill()

# Combine deposits_pivot and rewards_pivot by summing the values
staked_pool_size = deposits_pivot.add(rewards_pivot, fill_value=0)

# Subtract the values from withdrawals_pivot
staked_pool_size = staked_pool_size.subtract(withdrawals_pivot, fill_value=0)

staked_pool_size['total'] = staked_pool_size.sum(axis=1)

# Display the combined pivot table
staked_pool_size

pool_size_label,1,100+,2-5,20-99,6-19,total
slot,,,,,,
1200.0,NaN,NaN,NaN,NaN,0.102558,1.025580e-01
1500.0,NaN,9.992850e-01,NaN,NaN,0.102558,1.101843e+00
1800.0,NaN,9.992850e-01,NaN,NaN,0.172737,1.172022e+00
2100.0,NaN,1.501353e+00,NaN,NaN,0.172737,1.674090e+00
2400.0,NaN,1.768078e+00,NaN,NaN,0.172737,1.940815e+00
...,...,...,...,...,...,...
8984700.0,328530.661709,2.987657e+07,292226.056672,1.393791e+06,540865.972208,3.243198e+07
8985000.0,328530.661709,2.987668e+07,292226.056672,1.393791e+06,540865.972208,3.243210e+07
8985300.0,328530.661709,2.987680e+07,292226.056672,1.393791e+06,540865.972208,3.243222e+07


In [9]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(1200, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200

slots_df['category'] = np.nan
slots_df['category_cumulative'] = np.nan

# Define the pool_size_columns for the correct unique values in your dataset
columns = rewards['category'].unique()

# Assign values from columns to category column cyclically
slots_df['category'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Create the pivot table with category as columns and category_cumulative as values
slots_pivot = slots_df.pivot_table(index='slot', columns='category', values='category_cumulative', dropna=False)

deposits_raw = deposits_df.pivot_table(index='slot', columns='category', values='category_cumulative')
withdrawals_raw = withdrawals_df.pivot_table(index='slot', columns='category', values='category_cumulative')
rewards_raw = rewards_df.pivot_table(index='slot', columns='category', values='category_cumulative')

# Combine deposits_pivot to slots_pivot by slot index
deposits_pivot = slots_pivot.combine_first(deposits_raw).ffill()
withdrawals_pivot = slots_pivot.combine_first(withdrawals_raw).ffill()
rewards_pivot = slots_pivot.combine_first(rewards_raw).ffill()

# Combine deposits_pivot and rewards_pivot by summing the values
staked_category = deposits_pivot.add(rewards_pivot, fill_value=0)

# Subtract the values from withdrawals_pivot
staked_category = staked_category.subtract(withdrawals_pivot, fill_value=0)

staked_category['total'] = staked_category.sum(axis=1)

# Display the combined pivot table
staked_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
slot,,,,,,,
1200.0,NaN,NaN,NaN,0.117054,NaN,NaN,1.170537e-01
1500.0,NaN,NaN,NaN,0.188426,NaN,NaN,1.884261e-01
1800.0,NaN,NaN,NaN,0.188426,NaN,1.416741e+00,1.605167e+00
2100.0,4.070735e-01,NaN,NaN,0.188426,NaN,1.416741e+00,2.012240e+00
2400.0,4.070735e-01,NaN,NaN,0.188426,NaN,1.900291e+00,2.495791e+00
...,...,...,...,...,...,...,...
8984700.0,8.251830e+06,2.531301e+06,1.074935e+07,524829.745963,1.827924e+06,8.573853e+06,3.245909e+07
8985000.0,8.251863e+06,2.531301e+06,1.074935e+07,524829.745963,1.827924e+06,8.573853e+06,3.245912e+07
8985300.0,8.251863e+06,2.531301e+06,1.074964e+07,524829.745963,1.827924e+06,8.573853e+06,3.245941e+07


In [10]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(1200, 8986176)})

slots_df['pool'] = np.nan
slots_df['pool_cumulative'] = np.nan

# Define the pool_size_columns for the correct unique values in your dataset
columns = rewards['pool'].unique()

# Assign values from columns to pool column cyclically
slots_df['pool'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Create the pivot table with pool as columns and pool_cumulative as values
slots_pivot = slots_df.pivot_table(index='slot', columns='pool', values='pool_cumulative', dropna=False)

deposits_raw = deposits.pivot_table(index='slot', columns='pool', values='pool_cumulative')
withdrawals_raw = withdrawals.pivot_table(index='slot', columns='pool', values='pool_cumulative')
rewards_raw = rewards.pivot_table(index='slot', columns='pool', values='pool_cumulative')

# Combine deposits_pivot to slots_pivot by slot index
deposits_pivot = slots_pivot.combine_first(deposits_raw).ffill()
withdrawals_pivot = slots_pivot.combine_first(withdrawals_raw).ffill()
rewards_pivot = slots_pivot.combine_first(rewards_raw).ffill()

# Combine deposits_pivot and rewards_pivot by summing the values
staked_pool = deposits_pivot.add(rewards_pivot, fill_value=0)

# Subtract the values from withdrawals_pivot
staked_pool = staked_pool.subtract(withdrawals_pivot, fill_value=0)

staked_pool = staked_pool.clip(lower=0)

staked_pool['total'] = staked_pool.sum(axis=1)

staked_pool.index = ((staked_pool.index - 1200) // 300) * 300 + 1200

staked_pool = staked_pool.groupby(staked_pool.index).last()

# Display the combined pivot table
staked_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
slot,,,,,,,,,,,,
0.0,9.930880e+05,0.052556,2.101344e+06,NaN,1.145088e+06,5632.000000,4.163520e+06,NaN,55745.000000,2.439882e-01,236214.000000,2.965439e-01
300.0,NaN,0.093414,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,5.264605e-01,0.000000,6.198749e-01
600.0,NaN,0.142021,NaN,NaN,NaN,0.000636,NaN,NaN,NaN,8.210019e-01,0.000000,9.636584e-01
900.0,NaN,0.165077,NaN,NaN,NaN,0.000636,NaN,NaN,NaN,1.138048e+00,0.000727,1.304488e+00
1200.0,9.930880e+05,290080.209582,2.101344e+06,NaN,1.145088e+06,5632.000636,4.163520e+06,NaN,55745.000000,1.942013e+06,236214.000727,1.093272e+07
...,...,...,...,...,...,...,...,...,...,...,...,...
8984700.0,1.119103e+06,608101.212211,4.496284e+06,1.176222e+06,7.578910e+05,497165.284923,9.422995e+06,489338.076127,354488.286325,1.293849e+07,810532.317936,3.267061e+07
8985000.0,1.119110e+06,608103.908809,4.496305e+06,1.176228e+06,7.578937e+05,497167.058365,9.423029e+06,489340.754478,354489.623113,1.293911e+07,810535.879265,3.267131e+07
8985300.0,1.119115e+06,608107.471418,4.496354e+06,1.176233e+06,7.578968e+05,497169.739646,9.423067e+06,489342.974175,354491.852005,1.293983e+07,810539.880242,3.267215e+07


In [11]:
# Export tables
staked_pool_size.to_csv('../int/staked_pool_size.csv')
staked_category.to_csv('../int/staked_category.csv')
staked_pool.to_csv('../int/staked_pool.csv')